# Rotten Tomatoes Reviews Dataset Cleaning
This notebook cleans and prepares the dataset containing rotten tomatoes reviews for further data analysis.

In [90]:
import pandas as pd
import numpy as np
import re
reviews_df = pd.read_csv('../../data/rotten_tomatoes_reviews.csv')
print(reviews_df.info())
print(reviews_df.head())
print(reviews_df.describe(include='all'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1129887 entries, 0 to 1129886
Data columns (total 9 columns):
 #   Column                Non-Null Count    Dtype 
---  ------                --------------    ----- 
 0   rotten_tomatoes_link  1129887 non-null  object
 1   movie_title           1129887 non-null  object
 2   critic_name           1111366 non-null  object
 3   top_critic            1129887 non-null  bool  
 4   publisher_name        1129887 non-null  object
 5   review_type           1129887 non-null  object
 6   review_score          823985 non-null   object
 7   review_date           1129887 non-null  object
 8   review_content        1064109 non-null  object
dtypes: bool(1), object(8)
memory usage: 70.0+ MB
None
  rotten_tomatoes_link                                        movie_title  \
0            m/0814255  Percy Jackson & the Olympians: The Lightning T...   
1            m/0814255  Percy Jackson & the Olympians: The Lightning T...   
2            m/0814255  Percy 

## Dataset Overview
The Dataset contains data about reviews of movies from Rotten Tomatoes.
The data is divided in **9 columns**:
- **rotten_tomatoes_link**: The path of the movie on Rotten Tomatoes
- **movie_title**: the title of the movie
- **critic_name**: the name of the critic who wrote the review
- **top_critic**: A boolean value indicating if the critic is a top critic or not
- **publisher_name**: If the review is published by a publisher, this column contains the name of the publisher
- **review_type**: The type of review, it can be:
    - **Fresh**: If the review is positive
    - **Rotten**: If the review is negative
- **review_score**: The score the critique gave to the movie
- **review_date**: The date the review was published
- **review_content**: The text of the review

---

## Starting the data cleaning process
The goals are:
- **Rename columns**: to give more unique identifiable meaning
- **Handling duplicates and redundant data**: since ids have to be unique for each movie
- **Adding Rotten Tomatoes domain to links**: to make the links usable
- **Handling data types**: to make sure the data is in the right format
- **Analyze missing values**: to see if there are any missing values in the data
- **Explore categorical and text data**: to see if there are any outliers or wrong values
- **Normalize review scores**: to make sure the scores are in the same format
- **Analyze review scores**: to see if there are any outliers or wrong values

In [91]:
# 1. Rename Columns
reviews_df.rename(columns={'review_content': 'review_text'}, inplace=True)

# 1. Checking Duplicates
# Count all duplicate rows (excluding the first occurrence)
dup_count = reviews_df.duplicated().sum()
print(f"Total duplicates: {dup_count}")

# Drop exact duplicate rows and reset index
reviews_df = reviews_df.drop_duplicates(keep='first').reset_index(drop=True)

# 2. Adding Rotten Tomatoes Domain to links
reviews_df['rotten_tomatoes_link'] = 'https://www.rottentomatoes.com' + reviews_df['rotten_tomatoes_link']

# 3. Handle Data Types
reviews_df['rotten_tomatoes_link'] = reviews_df['rotten_tomatoes_link'].astype('string')
reviews_df['review_type'] = reviews_df['review_type'].astype('category')
reviews_df['movie_title'] = reviews_df['movie_title'].astype('string')
reviews_df['critic_name'] = reviews_df['critic_name'].astype('string')
reviews_df['publisher_name'] = reviews_df['publisher_name'].astype('string')
reviews_df['review_text'] = reviews_df['review_text'].astype('string')
reviews_df['review_date'] = (
    pd.to_datetime(
        reviews_df['review_date'], errors='coerce'
    )
    .dt.normalize()
)

# 4. Analyze Missing Values
print("Missing values per column:")
print(reviews_df.isnull().sum())
print("\nPercentage of missing values per column:")
print((reviews_df.isnull().sum() / len(reviews_df)) * 100)

# 5. Explore Categorical and Text Data
print("\nUnique values in 'review_type':")
print(reviews_df['review_type'].value_counts())

print("\nNumber of unique publishers:")
print(reviews_df['publisher_name'].nunique())

print("\nNumber of unique critics:")
print(reviews_df['critic_name'].nunique())

print("\nDataFrame info after initial processing:")
print(reviews_df.info())

print("\nFirst 5 rows after initial processing:")
print(reviews_df.head())

Total duplicates: 119471
Missing values per column:
rotten_tomatoes_link         0
movie_title                  0
critic_name              16446
top_critic                   0
publisher_name               0
review_type                  0
review_score            273571
review_date                  0
review_text              58498
dtype: int64

Percentage of missing values per column:
rotten_tomatoes_link     0.000000
movie_title              0.000000
critic_name              1.627646
top_critic               0.000000
publisher_name           0.000000
review_type              0.000000
review_score            27.075086
review_date              0.000000
review_text              5.789497
dtype: float64

Unique values in 'review_type':
review_type
Fresh     643835
Rotten    366581
Name: count, dtype: int64

Number of unique publishers:
2230

Number of unique critics:
11108

DataFrame info after initial processing:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1010416 entries, 0 to 101041

In [92]:
# 6. Normalize Review Scores
scores = reviews_df['review_score'].astype(str).str.strip()

def checking_review_scores():
    # 1. Patterns
    letter_pat   = r'^[A-F][+-]?$'
    number_pat   = r'^\d+(\.\d*)?$'
    fraction_pat = r'^(\d+(\.\d*)?)/(\d+(\.\d*)?)$'

    # 2. Masks
    is_letter   = scores.str.fullmatch(letter_pat, case=False)
    is_number   = scores.str.fullmatch(number_pat)
    is_fraction = scores.str.fullmatch(fraction_pat)

    # 3. Counts
    letters_count   = is_letter.sum()
    numbers_count   = is_number.sum()
    fractions_count = is_fraction.sum()

    # 4. Denominator breakdown
    denoms = scores[is_fraction].str.extract(fraction_pat)[2]
    denominator_counts = denoms.value_counts()

    print(f'Letter grades: {letters_count}')
    print(f'Standalone numbers: {numbers_count}')
    print(f'Fractions: {fractions_count}')
    print('\nDenominator counts:')
    print(denominator_counts)

checking_review_scores()

Letter grades: 115384
Standalone numbers: 1420
Fractions: 620038

Denominator counts:
2
5       319714
4       230623
10       62037
100       6164
6          910
20         393
8          102
3           25
2           23
11           9
1            8
5.5          6
50           2
70           2
54           2
40           2
3.5          2
00           1
1000         1
80           1
95           1
13           1
60           1
0            1
9            1
5.4          1
90           1
18           1
45           1
44           1
24           1
Name: count, dtype: int64


In [93]:
# Precompile fraction regex and define letter mapping
_fraction_re = re.compile(r'^\s*(\d+(?:\.\d*)?)\s*/\s*(\d+(?:\.\d*)?)\s*$')
_letter_map = {
    'A+': 5.0, 'A': 4.75, 'A-': 4.5,
    'B+': 4.25, 'B': 4.0,  'B-': 3.75,
    'C+': 3.5,  'C': 3.25, 'C-': 3.0,
    'D+': 2.75, 'D': 2.5,  'D-': 2.25,
    'F': 0.0,
}

def normalize_score_100_series(scores: pd.Series) -> pd.Series:
    s = scores.astype(str).str.strip().str.upper()
    result = pd.Series(np.nan, index=s.index, dtype=float)

    # 1. Letter grades → 0–5 then *20
    mask_letter = s.isin(_letter_map)
    result.loc[mask_letter] = s.loc[mask_letter].map(_letter_map) * 20

    # 2. Fractions → (num/den)*100
    fracs = s.str.extract(_fraction_re)
    num = pd.to_numeric(fracs[0], errors='coerce')
    den = pd.to_numeric(fracs[1], errors='coerce')
    mask_frac = den.notna() & (den != 0)
    result.loc[mask_frac] = (num[mask_frac] / den[mask_frac]) * 100

    # 3. Standalone numbers: infer scale
    rest = result.isna()
    nums = pd.to_numeric(s[rest], errors='coerce')
    # ≤5, ≤10, ≤100
    result.loc[rest & (nums <= 5)]   = nums[nums <= 5]   / 5   * 100
    result.loc[rest & (nums > 5) & (nums <= 10)]  = nums[(nums > 5) & (nums <= 10)]  / 10  * 100
    result.loc[rest & (nums > 10) & (nums <= 100)] = nums[(nums > 10) & (nums <= 100)]

    # 4. Clamp, round to nearest int, and cast to nullable Int64
    return result.clip(0, 100).round().astype('Int64')

# Apply to the DataFrame
reviews_df['review_score'] = normalize_score_100_series(reviews_df['review_score'])

In [94]:
print("\nValue counts of review_score now normalized (top 20):")
print(reviews_df['review_score'].value_counts(dropna=False).nlargest(20))

print("\nDescriptive statistics for review_score:")
print(reviews_df['review_score'].describe())


Value counts of review_score now normalized (top 20):
review_score
<NA>    273579
80      107262
60       98318
75       80481
50       73693
40       58636
70       48878
100      40352
62       40294
88       30973
90       27071
20       20078
85       19316
38       18357
65       14285
25       13556
30       10122
95        9065
0         6244
55        4264
Name: count, dtype: Int64

Descriptive statistics for review_score:
count     736837.0
mean     64.897426
std      20.850312
min            0.0
25%           50.0
50%           68.0
75%           80.0
max          100.0
Name: review_score, dtype: Float64


In [95]:
print(reviews_df.info())
print(reviews_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1010416 entries, 0 to 1010415
Data columns (total 9 columns):
 #   Column                Non-Null Count    Dtype         
---  ------                --------------    -----         
 0   rotten_tomatoes_link  1010416 non-null  string        
 1   movie_title           1010416 non-null  string        
 2   critic_name           993970 non-null   string        
 3   top_critic            1010416 non-null  bool          
 4   publisher_name        1010416 non-null  string        
 5   review_type           1010416 non-null  category      
 6   review_score          736837 non-null   Int64         
 7   review_date           1010416 non-null  datetime64[ns]
 8   review_text           951918 non-null   string        
dtypes: Int64(1), bool(1), category(1), datetime64[ns](1), string(5)
memory usage: 56.9 MB
None
                      rotten_tomatoes_link  \
0  https://www.rottentomatoes.comm/0814255   
1  https://www.rottentomatoes.comm/08142

In [96]:
reviews_df.to_csv('../../data/data-cleaned/rotten-tomatoes-reviews-cleaned.csv', index=False)
print("DataFrame saved to CSV.")

DataFrame saved to CSV.


In [ ]:
%reset -f